In [14]:
import torch
from pathlib import Path
import os
import json
import pandas as pd
from tqdm import tqdm
from model import DiagnosticModel
from train_utils import get_info
from collections import defaultdict
import numpy as np

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
results_dir = Path('../outputs')
device = 'cuda'

In [24]:
results_dict = defaultdict(list)

for sbdir in tqdm(os.listdir(results_dir )):
    sbdir_path = results_dir / sbdir
    
    model_path = sbdir_path / 'best_model.pth'
    params_path = sbdir_path / 'params.json'
    results_path = sbdir_path / 'test_results.json'

    with open(params_path, 'r') as f:
        params = json.load(f)

    if not os.path.exists(results_path):
        continue 
    with open(results_path, 'r') as f:
        results = json.load(f)

    checkpoint = torch.load(model_path, map_location=device)
    # model = DiagnosticModel(model_name = params['model'], in_channels = 1)
    # model = model.to(device)
    # model.load_state_dict(checkpoint['model_state_dict'])
    
    val_metric = params['val_metric']
    start_epoch = checkpoint['epoch']
    best_val_metric_score = checkpoint[f'val_{val_metric}']
    full_train = checkpoint['full_train']
    full_val = checkpoint['full_val']
    full_loss = checkpoint['full_loss']
    
    results_dict['exps'].append(sbdir)
    results_dict['auc'].append(float(results['auc']))
    results_dict[f"val_{val_metric}"].append(float(best_val_metric_score))

    for test_metric, test_value in get_info(np.array(results['val_cfvalues'])).items():
        results_dict[f'test_{test_metric}'].append(test_value)
    


100%|██████████| 17/17 [00:02<00:00,  7.01it/s]


In [26]:
sort_metric = ['auc', 'test_acc']

df = pd.DataFrame(results_dict)

df = df.sort_values(by=sort_metric, ascending=False)
df = df.reset_index(drop=True)
df


,exps,auc,val_acc,test_tn,test_fp,test_fn,test_tp,test_tpr,test_fpr,test_prec,test_recall,test_f1,test_acc
0,balance_o_download,0.840459,0.856585,0.898693,0.003268,0.075163,0.022876,0.233333,0.003623,0.875000,0.233333,0.368421,0.921569
1,balance_b_download,0.835326,0.840960,0.877451,0.024510,0.057190,0.040850,0.416667,0.027174,0.625000,0.416667,0.500000,0.918301
2,balance_b_download2,0.819746,0.843192,0.888889,0.013072,0.066993,0.031046,0.316667,0.014493,0.703704,0.316667,0.436782,0.919935
3,balance_o_download2,0.808137,0.849330,0.890523,0.011438,0.063725,0.034314,0.350000,0.012681,0.750000,0.350000,0.477273,0.924837
4,balance_w_download,0.807820,0.833705,0.880719,0.021242,0.060458,0.037582,0.383333,0.023551,0.638889,0.383333,0.479167,0.918301
5,balance_w,0.778019,0.794085,0.831699,0.070261,0.057190,0.040850,0.416667,0.077899,0.367647,0.416667,0.390625,0.872549
6,balance_o,0.772222,0.785156,0.826797,0.075163,0.065359,0.032680,0.333333,0.083333,0.303030,0.333333,0.317460,0.859477
7,balance_b,0.763829,0.760045,0.846405,0.055556,0.076797,0.021242,0.216667,0.061594,0.276596,0.216667,0.242991,0.867647
8,balance_w_download2,0.709164,0.833705,0.880719,0.021242,0.058824,0.039216,0.400000,0.023551,0.648649,0.400000,0.494845,0.919935
